In [ ]:
%load_ext autoreload
%autoreload 2

In [6]:
import os
import json
import requests
import openai
from apiKey import OPENAI_API_KEY
from openai import OpenAI

# File to store cached documentation
CACHE_FILE = "doc_cache.json"
client = OpenAI(api_key=OPENAI_API_KEY)

def load_cache():
    """Load cached documentation from a JSON file."""
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r") as f:
            return json.load(f)
    return {}

def save_cache(cache):
    """Save the cache dictionary to a JSON file."""
    with open(CACHE_FILE, "w") as f:
        json.dump(cache, f)

def fetch_documentation(url, cache):
    """
    Fetch documentation content from a URL.
    If available in the cache, use that instead.
    """
    if url in cache:
        print("Using cached documentation.")
        return cache[url]
    else:
        print("Fetching documentation from URL.")
        response = requests.get(url)
        if response.status_code == 200:
            cache[url] = response.text
            save_cache(cache)
            return response.text
        else:
            raise Exception(f"Failed to fetch URL content. Status code: {response.status_code}")

def ask_question_using_cache(doc_text, question, json = False):
    """
    Ask ChatGPT a question using the cached documentation.
    The cached documentation (or an excerpt of it) is sent along with the question.
    """
    # Truncate if necessary to avoid token limits
    doc_excerpt = doc_text
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant that uses provided documentation as a reference."},
        {"role": "user", "content": f"Here is the cached documentation:\n\n{doc_excerpt}"},
        {"role": "user", "content": question}
    ]
    
    chat = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            response_format={'type': 'json_object' if json else 'text'}
        )
    return chat.choices[0].message.content

if __name__ == "__main__":
    # Set your OpenAI API key
    openai.api_key = "YOUR_OPENAI_API_KEY"
    
    # URL for the documentation (example: the README for the 'requests' library)
    doc_url = "https://docs.scenic-lang.org/en/latest/tutorials/dynamics.html"
    
    # Load existing cache (or initialize an empty cache)
    cache = load_cache()
    
    # Fetch the documentation using the cache mechanism
    documentation = fetch_documentation(doc_url, cache)
    
    # Later, use the cached documentation to ask a question
    user_question = "Can you read the documentation and provide an example scenic behavior?"
    answer = ask_question_using_cache(documentation, user_question)
    
    print("ChatGPT's Response:")
    print(answer)
    # print(documentation)

Using cached documentation.
ChatGPT's Response:
Certainly! Below is an example of a Scenic behavior that defines a simple "Follow Lane" behavior for a car. This behavior incorporates some of the concepts discussed in the documentation, such as using an infinite loop to continuously issue actions and the use of the `take` statement to apply actions.

```scenic
behavior FollowLaneBehavior():
    while True:
        throttle, steering = compute_controls()
        take SetThrottleAction(throttle), SetSteerAction(steering)
```

In this example:

- **`FollowLaneBehavior`** is the name of the behavior defined.
- An infinite loop (`while True:`) allows the car to continuously follow the lane.
- The function `compute_controls()` is assumed to calculate the throttle and steering values needed based on the current state of the car and the environment (the implementation of this function would depend on the specific requirements of the scenario).
- The `take` statement is used to issue actions (`S

In [7]:
import os
from unity_utils import UnityTranslator
import json

dir = os.getcwd()
DATA_DIR = dir + '/data/give-and-go1-3'
demos = UnityTranslator.get_from(DATA_DIR)

Importing: 100%|██████████████████████████████████████| 3/3 [00:05<00:00,  1.86s/it]


In [8]:
def load_python_file_as_string(file_path: str) -> str:
    """
    Load the contents of a Python file as a string.

    Args:
        file_path (str): The path to the Python file.

    Returns:
        str: The contents of the file as a string.
    """
    with open(file_path, "r", encoding="utf-8") as file:
        return file.read()

In [1]:
from nlp_utils import Chat
from scenic_fc.api import api
from nlp_utils import *
from synth_utils import * 

d = demos[0]

system_instruction = """
    You are provided with (1) a set of videos of the soccer coach's demonstration where the video shows a top-down view of a soccer field,
    (2) a set of corresponding transcriptions of a soccer coach explanining how to play soccer in each video, 
    (3) a documentation on Scenic programming language,
    (4) a library of APIs modeling Scenic behaviors, and
    (5) a library of APIs modeling physical constraints.

    Your task is to output a Scenic program modeling the behavior of the coach, only using the provided behaviors and constraint APIs. 
"""

# URL for the documentation (example: the README for the 'requests' library)
doc_url = "https://docs.scenic-lang.org/en/latest/tutorials/dynamics.html"

# Load existing cache (or initialize an empty cache)
cache = load_cache()

# Fetch the documentation using the cache mechanism
documentation = fetch_documentation(doc_url, cache)

action_library = load_python_file_as_string('/Users/edward_kim/Desktop/narrated_demo/v2/baseline/baseline_behavior.scenic')
constraint_library = load_python_file_as_string('/Users/edward_kim/Desktop/narrated_demo/v2/baseline/baseline_api.py')
example_scenic = load_python_file_as_string('/Users/edward_kim/Desktop/narrated_demo/v2/baseline/example.txt')

feedback = """
    This is a feedback to the program you output from previous query. Update the program with the following feedback. 
    1. Wait() and ReceiveBall() are not supported behaviors. Only use the behaviors from the provided library. 
    2. First you need to get a possession of the ball. Not wait until you get the ball. 
"""

entries = [
    Chat.Entry(role='system', text=system_instruction),
    Chat.Entry(role='user', text= "documentation of Scenic programming language: " + documentation),
    Chat.Entry(role='user', text= "a library of Scenic behaviors: " + action_library),
    Chat.Entry(role='user', text= "a library of constraints APIs: " + constraint_library),
    Chat.Entry(role='user', text= "an example of how to write a scenic program: " +example_scenic)
    # Chat.Entry(role='user', text= "This is a Scenic program you output in my previous query: " +example_scenic),
    # Chat.Entry(role='user', text= "Feedback to fix the program: " +feedback),
]

for i, d in enumerate(demos):
    entries += [Chat.Entry(role='user', text= f"transcription#{str(i)} of the video#{str(i)}: " + d.language)]
    if d.video:
        print(f"video length: {len(d.video.frame_dir)}")
        entries += [Chat.Entry(role='user', text=f'Video#{str(i)} Image Frame index: {idx}', im=get_im(im_dir)) for idx, im_dir in enumerate(d.video.frame_dir)]

chat = Chat(client, model='o3-mini')
output = chat(entries, json=False)
print(output)
# response = json.loads(output)
# print(response)

NameError: name 'demos' is not defined

In [25]:
for d in demos:
    print(d.language)

Okay, the same give and go. Um, at first, get possession of the ball. [Coach receives or gets possession of the ball]Pass the ball to your teammate.[Coach passes to teammate] Then you wanna go around[teammate receives or gets possession of the ball] your opponent and receive the ball.[teammate passes to Coach] [Coach receives or gets possession of the ball]Now, the point is, uh, in this case, because the opponent is around the same height as the teammate, uh, you need to, uh, you can't position yourself too wide. Uh, you need to position yourself around here to make, to create an angle of pass.
Okay, so I'm gonna teach you give-and-go. Uh, there's an opponent in front, teammate to the left. So first, get possession of the ball[Coach receives or gets possession of the ball]  and, uh, pass the ball to your teammate.[Coach passes to teammate] [teammate receives or gets possession of the ball] And here, in this case, if the opponent i- is under the teammate, uh, then you want to... So b- i